In [16]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import locale
import seaborn as sns
import matplotlib.pyplot as plt
import re
import pyproj
import folium

In [17]:
fecha = '20260505'

In [18]:
#Datos de IRP_procesado

irp = pd.read_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Indicadores_general/{fecha}_irp_procesado.csv', encoding='latin', sep=';')

irp.head(2)

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_45272\2615019220.py:3: DtypeWarning: Columns (4,27,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  irp = pd.read_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Indicadores_general/{fecha}_irp_procesado.csv', encoding='latin', sep=';')


,Fecha,ConcesiÃ³n,Concesionario de OperaciÃ³n,Id LÃ­nea,LÃ­nea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,C.V,Intervalo_Teorico,TP26,franja,Tipo dia,Tipo_dia,Turno_ccz,Turno_Sup,reg_ccz,Supervisor
0,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,128,7.5,NaN,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS
1,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,128,7.5,NaN,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS


In [19]:
irp['Conductor'] = irp['Conductor'].replace('', 0).fillna(0).astype(int)

In [20]:
irp = irp[
    irp['Hora Llegada_minutos'].notna() &
    (irp['Hora Llegada_minutos'] != '')
]

irp.head(2)

,Fecha,ConcesiÃ³n,Concesionario de OperaciÃ³n,Id LÃ­nea,LÃ­nea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,C.V,Intervalo_Teorico,TP26,franja,Tipo dia,Tipo_dia,Turno_ccz,Turno_Sup,reg_ccz,Supervisor
0,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,128,7.5,NaN,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS
1,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,128,7.5,NaN,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS


In [21]:
#Crear tipo de adelanto segundo horas de llegada a paradero vs el tiempo ajustado (referencia)
irp['estado_adelanto'] = np.select(
    [
        irp['Hora Llegada_minutos'] < irp['Hora Referencia_minutos'], 
        irp['Hora Llegada_minutos'] == irp['Hora Referencia_minutos'],
        irp['Hora Llegada_minutos'] > irp['Hora Referencia_minutos']
    ],
    [
        'Adelanto',
        'A Tiempo',
        'Atrasado'
    ]
)

irp.head(2)

,Fecha,ConcesiÃ³n,Concesionario de OperaciÃ³n,Id LÃ­nea,LÃ­nea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Intervalo_Teorico,TP26,franja,Tipo dia,Tipo_dia,Turno_ccz,Turno_Sup,reg_ccz,Supervisor,estado_adelanto
0,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,7.5,NaN,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto
1,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,7.5,NaN,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto


In [22]:
#Diferencia de llagada a paradas
irp['Diferencia_Min'] = (
    irp['Hora Llegada_minutos'] - irp['Hora Referencia_minutos']
).abs()

irp.head(2)

,Fecha,ConcesiÃ³n,Concesionario de OperaciÃ³n,Id LÃ­nea,LÃ­nea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,TP26,franja,Tipo dia,Tipo_dia,Turno_ccz,Turno_Sup,reg_ccz,Supervisor,estado_adelanto,Diferencia_Min
0,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,NaN,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto,12.24
1,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,NaN,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto,1.98


In [23]:
irp['Diferencia_Min'] = (
    (irp['Hora Llegada_minutos'] - irp['Hora Referencia_minutos'])
    .abs()
    .round(0)
    .fillna(0)
    .astype(int)
)

irp.head(2)

,Fecha,ConcesiÃ³n,Concesionario de OperaciÃ³n,Id LÃ­nea,LÃ­nea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,TP26,franja,Tipo dia,Tipo_dia,Turno_ccz,Turno_Sup,reg_ccz,Supervisor,estado_adelanto,Diferencia_Min
0,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,NaN,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto,12
1,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,NaN,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto,2


In [24]:
irp['categoria_estado_adelanto'] = irp['estado_adelanto'].map({
    'Adelanto': 1,
    'A Tiempo': 0,
    'Atrasado': -1
})

irp.head(2)

,Fecha,ConcesiÃ³n,Concesionario de OperaciÃ³n,Id LÃ­nea,LÃ­nea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,franja,Tipo dia,Tipo_dia,Turno_ccz,Turno_Sup,reg_ccz,Supervisor,estado_adelanto,Diferencia_Min,categoria_estado_adelanto
0,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto,12,1
1,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,3,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto,2,1


In [25]:
#rango de adelantos

irp['rango_adelanto'] = np.select(
    [
        (irp['estado_adelanto'] == 'Adelanto') & (irp['Diferencia_Min'] >= 10) & (irp['Diferencia_Min'] < 20),
        (irp['estado_adelanto'] == 'Adelanto') & (irp['Diferencia_Min'] > 20),
        (irp['estado_adelanto'] == 'Adelanto') & (irp['Diferencia_Min'] <= 10)
    ],
    [
        'Entre 10 y 20 min',
        'Mayor a 20 min',
        'Menor o igual a 10 min'
    ],
    default=np.nan
)

irp.head(2)

,Fecha,ConcesiÃ³n,Concesionario de OperaciÃ³n,Id LÃ­nea,LÃ­nea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Tipo dia,Tipo_dia,Turno_ccz,Turno_Sup,reg_ccz,Supervisor,estado_adelanto,Diferencia_Min,categoria_estado_adelanto,rango_adelanto
0,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto,12,1,Entre 10 y 20 min
1,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,HÃ¡bil,H,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto,2,1,Menor o igual a 10 min


In [26]:
irp['Cantidad'] = 1

In [27]:
irp['tp26_neg'] = (
    irp['TP26']
    .astype(str)
    .str.strip()
    .str.capitalize()
    .map({'Negativo': 1})
    .fillna(0)
    .astype(int)
)

irp.head(2)

,Fecha,ConcesiÃ³n,Concesionario de OperaciÃ³n,Id LÃ­nea,LÃ­nea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Turno_ccz,Turno_Sup,reg_ccz,Supervisor,estado_adelanto,Diferencia_Min,categoria_estado_adelanto,rango_adelanto,Cantidad,tp26_neg
0,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto,12,1,Entre 10 y 20 min,1,0
1,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,ET3N,SUP AM,DORIS EDITH PATARROYO GORDO,CAMILO ANDRES LOPEZ PENAGOS,Adelanto,2,1,Menor o igual a 10 min,1,0


In [28]:
# 🔹 Conteo por rangos
irp['adelanto_10_20'] = np.where(
    (irp['estado_adelanto'] == 'Adelanto') &
    (irp['rango_adelanto'] == 'Entre 10 y 20 min'),
    1, 0
)

irp['adelanto_mayor_20'] = np.where(
    (irp['estado_adelanto'] == 'Adelanto') &
    (irp['rango_adelanto'] == 'Mayor a 20 min'),
    1, 0
)

irp['adelanto_menor_10'] = np.where(
    (irp['estado_adelanto'] == 'Adelanto') &
    (irp['rango_adelanto'] == 'Menor o igual a 10 min'),
    1, 0
)

# 🔹 Promedios por rangos
irp['dif_10_20'] = np.where(
    (irp['estado_adelanto'] == 'Adelanto') &
    (irp['rango_adelanto'] == 'Entre 10 y 20 min'),
    irp['Diferencia_Min'],
    np.nan
)

irp['dif_mayor_20'] = np.where(
    (irp['estado_adelanto'] == 'Adelanto') &
    (irp['rango_adelanto'] == 'Mayor a 20 min'),
    irp['Diferencia_Min'],
    np.nan
)

irp['dif_menor_10'] = np.where(
    (irp['estado_adelanto'] == 'Adelanto') &
    (irp['rango_adelanto'] == 'Menor o igual a 10 min'),
    irp['Diferencia_Min'],
    np.nan
)

# 🔹 Flags por estado
irp['cant_adelanto'] = np.where(irp['estado_adelanto'] == 'Adelanto', 1, 0)
irp['cant_atraso'] = np.where(irp['estado_adelanto'] == 'Atrasado', 1, 0)
irp['cant_a_tiempo'] = np.where(irp['estado_adelanto'] == 'A Tiempo', 1, 0)

# 🔹 Diferencias por estado
irp['dif_adelanto'] = np.where(irp['estado_adelanto'] == 'Adelanto', irp['Diferencia_Min'], np.nan)
irp['dif_atraso'] = np.where(irp['estado_adelanto'] == 'Atrasado', irp['Diferencia_Min'], np.nan)

# 🔹 TP26 negativo (CORREGIDO)
irp['tp26_neg'] = (
    irp['TP26']
    .astype(str)
    .str.strip()
    .str.capitalize()
    .map({'Negativo': 1})
    .fillna(0)
    .astype(int)
)

irp.head(2)

,Fecha,ConcesiÃ³n,Concesionario de OperaciÃ³n,Id LÃ­nea,LÃ­nea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,adelanto_mayor_20,adelanto_menor_10,dif_10_20,dif_mayor_20,dif_menor_10,cant_adelanto,cant_atraso,cant_a_tiempo,dif_adelanto,dif_atraso
0,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,0,0,12.0,NaN,NaN,1,0,0,12.0,NaN
1,2026-05-05,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,0,1,NaN,NaN,2.0,1,0,0,2.0,NaN


In [29]:

resultado_resumido = (
    irp
    .groupby([
        'Fecha','NÃºmero FMS Bus','Servicio Bus','Id LÃ­nea',
        'LÃ­nea','Ruta','Id Ruta','Tabla','Conductor','Id Viaje',
        'Nombre de Conductor','reg_ccz','Supervisor'
    ], as_index=False)
    .agg({
        'Cantidad': 'sum',
        'cant_adelanto': 'sum',
        'cant_atraso': 'sum',
        'cant_a_tiempo': 'sum',
        'dif_adelanto': 'mean',
        'dif_atraso': 'mean',
        'tp26_neg': 'sum',
        'adelanto_10_20': 'sum',
        'adelanto_mayor_20': 'sum',
        'adelanto_menor_10': 'sum',
        'dif_10_20': 'mean',
        'dif_mayor_20': 'mean',
        'dif_menor_10': 'mean'
    })
)

resultado_resumido[['dif_adelanto', 'dif_atraso']] = (
    resultado_resumido[['dif_adelanto', 'dif_atraso']]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .round(0)
    .astype(int)
)

resultado_resumido[['dif_10_20', 'dif_mayor_20','dif_menor_10']] = (
    resultado_resumido[['dif_10_20', 'dif_mayor_20','dif_menor_10']]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .round(0)
    .astype(int)
)

resultado_resumido.head(2)

,Fecha,NÃºmero FMS Bus,Servicio Bus,Id LÃ­nea,LÃ­nea,Ruta,Id Ruta,Tabla,Conductor,Id Viaje,...,cant_a_tiempo,dif_adelanto,dif_atraso,tp26_neg,adelanto_10_20,adelanto_mayor_20,adelanto_menor_10,dif_10_20,dif_mayor_20,dif_menor_10
0,2026-05-05,0,CE166G027,10304,806,806_Ida_V3,12779,38,0,3,...,0,0,1,5,0,0,0,0,0,0
1,2026-05-05,502002,CE1770007,10232,359,359_V2,11181,7,510486,3,...,0,5,2,76,6,0,85,10,0,4


In [30]:
resultado_resumido.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/IRP_adelantos/{fecha}_irp_adelantos.csv', index=False, sep=';')